# LANDSAFE NER — AI Landslide Risk Prediction & Model Training Pipeline
### SIH 2026 (Problem Statement ID: SIH26001)
**Decoupled Machine Learning Pipeline for North Eastern Region Hazard Monitoring**

This notebook trains and exports the tabular landslide risk prediction models using XGBoost, Scikit-Learn, and ONNX Runtime for edge and cloud deployment.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
import xgboost as xgb
import joblib

print('All ML libraries loaded successfully!')

## 1. Exploratory Data Analysis & Hydrometeorological Features
Load synthetic geological & precipitation records calibrated with GSI/IMD regional thresholds.

In [ ]:
dataset_path = '../datasets/ner_landslide_historical_dataset.csv'
df = pd.read_csv(dataset_path)
print(f'Total records: {len(df)}')
df.head()

In [ ]:
df.describe().T

## 2. Model Training: XGBoost Landslide Risk Predictor

In [ ]:
feature_cols = [
    'rainfall_24h_mm',
    'rainfall_72h_mm',
    'soil_moisture_pct',
    'slope_angle_deg',
    'vulnerability_index',
    'pore_water_pressure_kpa',
    'elevation_m',
    'vegetation_ndvi'
]
target_col = 'landslide_occurred'

X = df[feature_cols].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print('AUC Score:', roc_auc_score(y_test, y_prob))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## 3. ONNX Model Export for Cloud & Offline Mobile Edge

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, len(feature_cols)]))]
onnx_model = convert_sklearn(model, initial_types=initial_type, target_opset=12)
with open('../export/risk_model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())
print('Exported model to ../export/risk_model.onnx')